## IMPORTS

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 13.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 341kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.17MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.0MB/s]


## Teacher Model

In [5]:
class TeacherMLP(nn.Module):
    def __init__(self, hidden1=512, hidden2=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 10)
        )

    def forward(self, x):
        return self.net(x)

teacher = TeacherMLP().to(device)

## Train Teacher

In [6]:
def train_teacher(model, loader, epochs=3, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Teacher Epoch {epoch+1}: Loss = {total_loss/len(loader):.4f}")

train_teacher(teacher, train_loader)

Teacher Epoch 1: Loss = 0.2955
Teacher Epoch 2: Loss = 0.1380
Teacher Epoch 3: Loss = 0.1035


## Freeze Teacher

In [7]:
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

## Student Model

In [8]:
class StudentMLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10)
        )

    def forward(self, x):
        return self.net(x)

student = StudentMLP().to(device)

## Distillation Training

In [9]:
temperature = 2.0
alpha = 0.7

ce_loss = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction="batchmean")

optimizer = optim.Adam(student.parameters(), lr=1e-3)

In [10]:
def distill(student, teacher, loader, epochs=3):
    for epoch in range(epochs):
        student.train()
        total_loss = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)

            # ----- Teacher Forward (no grad) -----
            with torch.no_grad():
                teacher_logits = teacher(x)
                teacher_probs = F.softmax(teacher_logits / temperature, dim=1)

            # ----- Student Forward -----
            student_logits = student(x)
            student_log_probs = F.log_softmax(student_logits / temperature, dim=1)

            # ----- Soft Loss (KL) -----
            loss_soft = kl_loss(student_log_probs, teacher_probs) * (temperature ** 2)

            # ----- Hard Loss (CE) -----
            loss_hard = ce_loss(student_logits, y)

            # ----- Total Loss -----
            loss = alpha * loss_soft + (1 - alpha) * loss_hard

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Student Epoch {epoch+1}: Loss = {total_loss/len(loader):.4f}")

distill(student, teacher, train_loader)

Student Epoch 1: Loss = 0.8383
Student Epoch 2: Loss = 0.2914
Student Epoch 3: Loss = 0.1669


## Evaluation

In [11]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

## Compare

In [12]:
teacher_acc = evaluate(teacher, test_loader)
student_acc = evaluate(student, test_loader)

print(f"Teacher Accuracy: {teacher_acc:.2f}%")
print(f"Student Accuracy: {student_acc:.2f}%")

Teacher Accuracy: 96.79%
Student Accuracy: 95.96%
